# Predikcia dojazdu elektrických áut: preprocessing

Pokračujeme z notebooku `analyza.ipynb`, ktorý vyčistil dáta a uložil ich do `data/ev_clean.csv`.
Tu dáta pripravíme na trénovanie modelu.

## 1. Načítanie vyčistených dát

In [1]:
import pandas as pd

data = pd.read_csv('data/ev_clean.csv')
print(data.shape)
data.head()

(1585, 9)


,year,make,vehicle_class,drive,motor_kw,n_motors,charge_time_240v_hr,kwh_per_100km,range_km
0,2011,Nissan,Midsize Cars,Front-Wheel Drive,80.0,1,7.0,21.13,117.0
1,2011,smart,Two Seaters,Rear-Wheel Drive,30.0,1,8.0,24.23,101.0
2,2011,smart,Two Seaters,Rear-Wheel Drive,30.0,1,8.0,24.23,101.0
3,2012,Mitsubishi,Subcompact Cars,Rear-Wheel Drive,49.0,1,7.0,18.64,100.0
4,2012,Azure Dynamics,Special Purpose Vehicle 2WD,Front-Wheel Drive,52.0,1,8.0,33.55,90.0


In [2]:
# kontrola, že čistenie prebehlo: žiadne chýbajúce hodnoty
print('Chýbajúce hodnoty:', data.isna().sum().sum())

Chýbajúce hodnoty: 0


## 2. Rozdelenie na X a y

- **`y`** je cieľová premenná, ktorú chceme predpovedať: dojazd `range_km`.
- **`X`** sú príznaky, z ktorých ju model predpovedá: všetky ostatné stĺpce.

In [3]:
y = data['range_km']
X = data.drop(columns='range_km')

print('X:', X.shape)
print('y:', y.shape)

X: (1585, 8)
y: (1585,)


In [4]:
# príznaky rozdelíme podľa typu, každý typ budeme spracovávať inak
num_col = X.select_dtypes(include='number').columns.tolist()
cat_col = X.select_dtypes(exclude='number').columns.tolist()

print('Číselné príznaky:', num_col)
print('Kategorické príznaky:', cat_col)

Číselné príznaky: ['year', 'motor_kw', 'n_motors', 'charge_time_240v_hr', 'kwh_per_100km']
Kategorické príznaky: ['make', 'vehicle_class', 'drive']


In [5]:
X.head()

,year,make,vehicle_class,drive,motor_kw,n_motors,charge_time_240v_hr,kwh_per_100km
0,2011,Nissan,Midsize Cars,Front-Wheel Drive,80.0,1,7.0,21.13
1,2011,smart,Two Seaters,Rear-Wheel Drive,30.0,1,8.0,24.23
2,2011,smart,Two Seaters,Rear-Wheel Drive,30.0,1,8.0,24.23
3,2012,Mitsubishi,Subcompact Cars,Rear-Wheel Drive,49.0,1,7.0,18.64
4,2012,Azure Dynamics,Special Purpose Vehicle 2WD,Front-Wheel Drive,52.0,1,8.0,33.55


In [6]:
y.describe().round(1)

count    1585.0
mean      447.8
std       120.7
min        61.0
25%       389.0
50%       455.0
75%       507.0
max       837.0
Name: range_km, dtype: float64

V `X` je 8 príznakov: 5 číselných (`year`, `motor_kw`, `n_motors`, `charge_time_240v_hr`,
`kwh_per_100km`) a 3 kategorické (`make`, `vehicle_class`, `drive`). Kategorické príznaky
v ďalšom kroku zakódujeme cez one-hot encoding, číselné podľa typu modelu škálujeme.

## 3. Rozdelenie na trénovacie a testovacie dáta

Dáta rozdelíme **ešte pred** kódovaním a škálovaním. Všetko, čo sa z dát niečo učí (priemer a smerodajná
odchýlka pre scaler, zoznam kategórií pre encoder), sa musí naučiť **iba na trénovacích dátach**.
Testovacie dáta predstavujú úplne nové autá, ktoré model nikdy nevidel. Keby sa z nich niečo dostalo
do tréningu (*data leakage*), výsledok na teste by vyšiel lepší, ako je v skutočnosti.

- **`test_size=0.2`:** 80 % áut na tréning, 20 % na test.
- **`shuffle=True`:** dáta sú zoradené podľa roku, bez premiešania by v teste boli iba najnovšie autá.
- **`random_state=42`:** rozdelenie bude pri každom spustení rovnaké, takže výsledky sa dajú zopakovať.

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42)

print('Tréning:', X_train.shape, y_train.shape)
print('Test:   ', X_test.shape, y_test.shape)

Tréning: (1268, 8) (1268,)
Test:    (317, 8) (317,)


In [8]:
# kontrola, či majú obe časti podobné rozdelenie dojazdu aj rokov
pd.DataFrame({
    'dojazd tréning': y_train.describe(),
    'dojazd test': y_test.describe(),
    'rok tréning': X_train['year'].describe(),
    'rok test': X_test['year'].describe(),
}).round(1)

,dojazd tréning,dojazd test,rok tréning,rok test
count,1268.0,317.0,1268.0,317.0
mean,447.9,447.1,2023.7,2023.6
std,119.4,126.0,3.2,3.0
min,61.0,92.0,2011.0,2012.0
25%,390.5,389.0,2023.0,2023.0
50%,455.0,452.0,2025.0,2024.0
75%,509.0,505.0,2026.0,2026.0
max,830.0,837.0,2027.0,2027.0


Priemer, medián aj rozptyl dojazdu a rokov sú v oboch častiach podobné, takže test je
reprezentatívna vzorka a nie napríklad iba nové autá.

## 4. Kódovanie a škálovanie

Modely vedia počítať iba s číslami, preto treba kategorické príznaky previesť na čísla.

**One-hot encoding:** z každej kategórie vznikne samostatný stĺpec s hodnotou 0 alebo 1.
Napríklad `drive = 'Rear-Wheel Drive'` sa zmení na `drive_Rear-Wheel Drive = 1` a ostatné stĺpce pohonu budú 0.
Nepoužívame očíslovanie kategórií (Tesla = 1, BMW = 2…), lebo model by si myslel, že BMW je „viac“ ako Tesla.

Nastavenia `OneHotEncoder`:

- **`min_frequency=10`:** značky a triedy, ktoré sú v trénovacích dátach menej ako 10-krát, zlúči do
  jedného stĺpca `infrequent`. V dátach je veľa značiek s 1–7 autami (Kandi, Scion, Fisker…),
  z ktorých by sa model aj tak nič nenaučil, len by pribudli zbytočné stĺpce.
- **`handle_unknown='infrequent_if_exist'`:** ak sa v teste objaví kategória, ktorú tréning nevidel,
  zaradí ju medzi zriedkavé namiesto toho, aby skončil chybou.
- **`sparse_output=False`:** výsledok bude bežná tabuľka, nie riedka matica, aby sa dala ľahko prezrieť.

**Škálovanie:** `StandardScaler` prepočíta každý číselný príznak tak, aby mal priemer 0 a smerodajnú
odchýlku 1. Potrebujú ho modely, ktoré porovnávajú veľkosti čísel (lineárna regresia s regularizáciou,
KNN, SVR), inak by `motor_kw` (stovky) prevážil `n_motors` (1–4). Stromové modely sa pýtajú iba
„je hodnota väčšia ako hranica?“, takže škálovanie nepotrebujú.

Preto pripravíme **dva** `ColumnTransformer`y. Oba kódujú kategórie rovnako, líšia sa iba tým,
či číselné stĺpce škálujú.

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def novy_encoder():
    return OneHotEncoder(min_frequency=10, handle_unknown='infrequent_if_exist',
                         sparse_output=False)

# pre lineárne modely, KNN a SVR: čísla škálujeme
prep_skalovany = ColumnTransformer([
    ('num', StandardScaler(), num_col),
    ('cat', novy_encoder(), cat_col),
])

# pre stromové modely (Random Forest, Gradient Boosting): čísla necháme bez zmeny
prep_stromy = ColumnTransformer([
    ('num', 'passthrough', num_col),
    ('cat', novy_encoder(), cat_col),
])

### 4.1 Ukážka výsledku

Pri trénovaní modelov dáme transformer do `Pipeline` spolu s modelom a ten ho naučí sám. Tu ho
naučíme ručne, len aby sme videli, čo z dát vyrobí.

- **`fit_transform` na tréningu:** naučí sa priemery, odchýlky a zoznam kategórií a hneď dáta prevedie.
- **`transform` na teste:** iba použije to, čo sa naučil na tréningu. Na teste sa nič neučí.

In [10]:
Xt_train = prep_skalovany.fit_transform(X_train)
Xt_test = prep_skalovany.transform(X_test)

print('Pred transformáciou:', X_train.shape)
print('Po transformácii:   ', Xt_train.shape, Xt_test.shape)

Pred transformáciou: (1268, 8)
Po transformácii:    (1268, 49) (317, 49)


In [11]:
# výsledok ako tabuľka s názvami stĺpcov
Xt_train = pd.DataFrame(Xt_train, columns=prep_skalovany.get_feature_names_out(), index=X_train.index)
Xt_train.head()

,num__year,num__motor_kw,num__n_motors,num__charge_time_240v_hr,num__kwh_per_100km,cat__make_Audi,cat__make_BMW,cat__make_Cadillac,cat__make_Chevrolet,cat__make_Dodge,...,cat__vehicle_class_Standard Sport Utility Vehicle 2WD,cat__vehicle_class_Standard Sport Utility Vehicle 4WD,cat__vehicle_class_Subcompact Cars,cat__vehicle_class_Two Seaters,cat__vehicle_class_infrequent_sklearn,cat__drive_4-Wheel Drive,cat__drive_All-Wheel Drive,cat__drive_Front-Wheel Drive,cat__drive_Part-time 4-Wheel Drive,cat__drive_Rear-Wheel Drive
552,0.103583,-0.833106,-1.336009,-0.897719,-0.737141,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1247,0.721666,-0.664260,0.387347,-0.332396,-0.609953,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1213,0.721666,1.092983,0.387347,0.609808,0.324220,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1047,0.412624,-0.582964,0.387347,0.609808,1.247428,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
277,-0.514501,1.612026,0.387347,0.986690,-1.267821,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [12]:
# ktoré kategórie encoder zlúčil do 'infrequent'
encoder = prep_skalovany.named_transformers_['cat']
for stlpec, zriedkave in zip(cat_col, encoder.infrequent_categories_):
    pocet = 0 if zriedkave is None else len(zriedkave)
    print(f'{stlpec}: {len(encoder.categories_[cat_col.index(stlpec)])} kategórií, zlúčených {pocet}')
    if zriedkave is not None:
        print('   ', list(zriedkave))

make: 41 kategórií, zlúčených 15
    ['Acura', 'Azure Dynamics', 'BYD', 'Bugatti Rimac', 'CODA Automotive', 'Fiat', 'Fisker', 'Jaguar', 'Jeep', 'Lordstown', 'MINI', 'Maserati', 'Mazda', 'Mitsubishi', 'Scion']
vehicle_class: 18 kategórií, zlúčených 7
    ['Midsize Station Wagons', 'Minicompact Cars', 'Minivan - 2WD', 'Special Purpose Vehicle 2WD', 'Special Purpose Vehicle 4WD', 'Sport Utility Vehicle - 2WD', 'Standard Pickup Trucks 2WD']
drive: 5 kategórií, zlúčených 0


In [13]:
# kontrola škálovania: na tréningu priemer 0 a odchýlka 1,
# na teste iba približne, lebo scaler použil priemery z tréningu
Xt_test = pd.DataFrame(Xt_test, columns=prep_skalovany.get_feature_names_out(), index=X_test.index)
skalovane = ['num__' + c for c in num_col]
pd.DataFrame({
    'priemer tréning': Xt_train[skalovane].mean(),
    'std tréning': Xt_train[skalovane].std(ddof=0),
    'priemer test': Xt_test[skalovane].mean(),
    'std test': Xt_test[skalovane].std(ddof=0),
}).round(2)

,priemer tréning,std tréning,priemer test,std test
num__year,-0.0,1.0,-0.01,0.94
num__motor_kw,-0.0,1.0,0.01,1.15
num__n_motors,-0.0,1.0,-0.01,1.19
num__charge_time_240v_hr,-0.0,1.0,0.03,1.05
num__kwh_per_100km,0.0,1.0,-0.02,1.06


- Z 8 príznakov vzniklo vyše 40 stĺpcov, väčšinu tvoria značky a triedy vozidiel.
- Zriedkavé značky a triedy sú zlúčené do jedného stĺpca, takže model nemá stĺpce pre značky s jedným autom.
- Na tréningu majú číselné stĺpce presne priemer 0 a odchýlku 1. Na teste sú hodnoty len blízko,
  čo je správne: scaler pri teste nepočítal nové priemery, iba použil tie z tréningu.

## 5. Ďalší krok: modely

Preprocessing je pripravený. V ďalšej časti:

1. **Baseline:** `DummyRegressor`, ktorý vždy predpovedá priemerný dojazd. Každý model ho musí prekonať.
2. **Modely v `Pipeline`:** `prep_skalovany` + `Ridge`, `prep_stromy` + `RandomForestRegressor`
   a `GradientBoostingRegressor`.
3. **Porovnanie** cez krížovú validáciu na trénovacích dátach, metriky MAE (chyba v km) a R².
4. **Ladenie** najlepšieho modelu cez `GridSearchCV`.
5. **Finálne vyhodnotenie** na testovacích dátach iba raz, na víťaznom modeli.